<div style="background: linear-gradient(135deg, #1a472a 0%, #0d1117 50%, #1a2a1a 100%); padding: 30px; border-radius: 16px; border: 2px solid #3fb950; font-family: 'Segoe UI', sans-serif;">
  <h1 style="color: #3fb950; font-size: 2.2em; margin: 0 0 8px 0;">🌿 Deep Learning for Plant Disease Detection</h1>
  <h3 style="color: #58a6ff; margin: 0 0 16px 0;">Using Image Processing & Convolutional Neural Networks</h3>
  <div style="display: flex; gap: 20px; flex-wrap: wrap;">
    <span style="background:#161b22; color:#8b949e; padding:6px 14px; border-radius:20px; border:1px solid #30363d;">🔥 PyTorch + CUDA</span>
    <span style="background:#161b22; color:#8b949e; padding:6px 14px; border-radius:20px; border:1px solid #30363d;">📦 New Plant Diseases Dataset</span>
    <span style="background:#161b22; color:#8b949e; padding:6px 14px; border-radius:20px; border:1px solid #30363d;">🏗️ Custom CNN + EfficientNetB0</span>
    <span style="background:#161b22; color:#8b949e; padding:6px 14px; border-radius:20px; border:1px solid #30363d;">📊 Grad-CAM · Confusion Matrix</span>
  </div>
</div>

---

## 📋 Table of Contents
| # | Section |
|---|---|
| 1 | [Setup & Environment Check](#setup) |
| 2 | [Dataset Exploration & EDA](#eda) |
| 3 | [Data Augmentation Preview](#aug) |
| 4 | [DataLoaders](#loaders) |
| 5 | [Model A — Custom CNN](#cnn) |
| 6 | [Model B — EfficientNet-B0 (Transfer Learning)](#effnet) |
| 7 | [Training Engine](#train) |
| 8 | [Train Both Models](#run) |
| 9 | [Accuracy & Loss Curves](#curves) |
| 10 | [Confusion Matrix](#cm) |
| 11 | [Per-Class F1 Analysis](#f1) |
| 12 | [Prediction Gallery](#preds) |
| 13 | [Grad-CAM Heatmaps](#gradcam) |
| 14 | [Final Summary Dashboard](#summary) |
| 15 | [Single Image Inference](#inference) |

---
## 1. ⚙️ Setup & Environment Check <a id='setup'></a>

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  INSTALL (run once, then comment out)                           ║
# ╚══════════════════════════════════════════════════════════════════╝
#!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
#!pip install timm matplotlib seaborn scikit-learn opencv-python pillow tqdm ipywidgets

# ── Standard imports ──────────────────────────────────────────────────────────
import os, sys, time, random, warnings, json
from pathlib import Path
from collections import defaultdict
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import cv2
from PIL import Image
from tqdm.notebook import tqdm
from IPython.display import display, HTML, clear_output

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, OneCycleLR
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
import torchvision.models as tv_models

# ── timm for EfficientNet ─────────────────────────────────────────────────────
try:
    import timm
    HAS_TIMM = True
except ImportError:
    HAS_TIMM = False
    print("⚠️  timm not found — will use torchvision EfficientNet instead")

# ── sklearn ───────────────────────────────────────────────────────────────────
from sklearn.metrics import (
    classification_report,confusion_matrix,accuracy_score,f1_score,precision_score,
    recall_score,cohen_kappa_score,roc_auc_score,matthews_corrcoef
)

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = False  # set True for reproducibility, but may slow down training
torch.backends.cudnn.benchmark     = True  # set True for speed if input size is fixed

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Dark plot theme ───────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor':  '#161b22',
    'axes.edgecolor':   '#30363d', 'axes.labelcolor': '#e6edf3',
    'xtick.color':      '#8b949e', 'ytick.color':     '#8b949e',
    'text.color':       '#e6edf3', 'grid.color':      '#21262d',
    'grid.linewidth':   0.8,       'font.family':     'DejaVu Sans',
    'font.size':        11,        'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d',
})
C = dict(green='#3fb950', yellow='#d29922', red='#f85149',
         blue='#58a6ff', purple='#bc8cff', orange='#ff7b50',
         teal='#39d353', pink='#f778ba')

# ── Environment banner ────────────────────────────────────────────────────────
cuda_info = ''
if torch.cuda.is_available():
    cuda_info = f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB"

display(HTML(f"""
<div style="background:#0d1117;padding:18px;border-radius:12px;
            border:1px solid #3fb950;font-family:monospace;">
  <b style="color:#3fb950;font-size:16px;">✅ Environment Ready</b><br><br>
  <span style="color:#58a6ff;">PyTorch :</span> <span style="color:#e6edf3;">{torch.__version__}</span><br>
  <span style="color:#58a6ff;">Device  :</span> <span style="color:#{'3fb950' if str(DEVICE)=='cuda' else 'f85149'};"
  >{str(DEVICE).upper()}</span>  {cuda_info}<br>
  <span style="color:#58a6ff;">timm    :</span> <span style="color:#e6edf3;">{timm.__version__ if HAS_TIMM else 'not installed'}</span>
</div>
"""))
print(f"\n🌿 Device  : {DEVICE}")
print(f"🔢 Seed    : {SEED}")

---
## 2. 🔍 Dataset Exploration & EDA <a id='eda'></a>

> **Dataset location:** Unzip the Kaggle dataset so the folder tree looks like:
> ```
> data/
>   New Plant Diseases Dataset(Augmented)/
>     New Plant Diseases Dataset(Augmented)/
>       train/   ← set TRAIN_DIR to this
>       valid/   ← set VALID_DIR to this
> ```
> **Quick download (if kaggle CLI is set up):**  
> `kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p ./data --unzip`

In [ ]:
# ── Locate dataset ────────────────────────────────────────────────────────────
BASE = Path(r"C:\Users\Kartik\plant_disease_data\New Plant Diseases Dataset(Augmented)\New Plant Diseases Dataset(Augmented)")
TRAIN_DIR = VALID_DIR = None

search_roots = [
    BASE / 'New Plant Diseases Dataset(Augmented)' / 'New Plant Diseases Dataset(Augmented)',
    BASE / 'new-plant-diseases-dataset' / 'New Plant Diseases Dataset(Augmented)' / 'New Plant Diseases Dataset(Augmented)',
    BASE,
]
for root in search_roots:
    if (root / r"C:\Users\Kartik\plant_disease_data\New Plant Diseases Dataset(Augmented)\New Plant Diseases Dataset(Augmented)\train").exists():
        TRAIN_DIR = root / r"C:\Users\Kartik\plant_disease_data\New Plant Diseases Dataset(Augmented)\New Plant Diseases Dataset(Augmented)\train"
        VALID_DIR = root / r"C:\Users\Kartik\plant_disease_data\New Plant Diseases Dataset(Augmented)\New Plant Diseases Dataset(Augmented)\valid"
        break

if TRAIN_DIR is None:
    raise FileNotFoundError(
        "Cannot locate train/ directory.\n"
        "Please unzip the Kaggle dataset into ./data/ and re-run this cell."
    )

print(f"📂 Train : {TRAIN_DIR}")
print(f"📂 Valid : {VALID_DIR}")

# ── Class inventory ───────────────────────────────────────────────────────────
CLASS_NAMES  = sorted([d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])
NUM_CLASSES  = len(CLASS_NAMES)
cls2idx      = {c: i for i, c in enumerate(CLASS_NAMES)}

EXT = ('*.jpg','*.JPG','*.jpeg','*.png','*.PNG')
def count_imgs(folder):
    return sum(len(list(folder.glob(e))) for e in EXT)

train_counts = {c: count_imgs(TRAIN_DIR / c) for c in CLASS_NAMES}
valid_counts = {c: count_imgs(VALID_DIR / c) for c in CLASS_NAMES}
total_train  = sum(train_counts.values())
total_valid  = sum(valid_counts.values())

plants  = sorted(set(c.split('___')[0] for c in CLASS_NAMES))
healthy = [c for c in CLASS_NAMES if 'healthy' in c.lower()]

print(f"\n{'='*50}")
print(f"  📊 Dataset Summary")
print(f"{'='*50}")
print(f"  Classes        : {NUM_CLASSES}")
print(f"  Plant species  : {len(plants)}")
print(f"  Healthy cls    : {len(healthy)}")
print(f"  Diseased cls   : {NUM_CLASSES - len(healthy)}")
print(f"  Train images   : {total_train:,}")
print(f"  Valid images   : {total_valid:,}")
print(f"  Total images   : {total_train+total_valid:,}")
print(f"  Avg Images/Class : {(total_train/NUM_CLASSES):.0f}")
print(f"{'='*50}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — Dataset Distribution Dashboard
# ══════════════════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(22, 14), facecolor='#0d1117')
fig.suptitle('🌿 Plant Disease Dataset — Exploratory Analysis',
             fontsize=18, color=C['green'], fontweight='bold', y=0.98)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── 1a. Images per plant species ──────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
sorted_cls  = sorted(CLASS_NAMES, key=lambda c: train_counts[c], reverse=True)
bar_colors  = [C['green'] if 'healthy' in c.lower() else C['yellow'] for c in sorted_cls]
x_pos       = np.arange(len(sorted_cls))
bars        = ax1.bar(x_pos, [train_counts[c] for c in sorted_cls],
                      color=bar_colors, edgecolor='#0d1117', linewidth=0.4, width=0.8)
ax1.set_xticks(x_pos)
ax1.set_xticklabels(
    [c.replace('___','\n').replace('_',' ') for c in sorted_cls],
    rotation=90, fontsize=6.5, ha='center'
)
ax1.set_ylabel('Training Images')
ax1.set_title('Training Images per Class', color=C['blue'], fontsize=13, pad=8)
ax1.grid(axis='y', alpha=0.3)
ax1.set_xlim(-0.8, len(sorted_cls) - 0.2)
p1 = mpatches.Patch(color=C['green'],  label='Healthy')
p2 = mpatches.Patch(color=C['yellow'], label='Diseased')
ax1.legend(handles=[p1, p2], loc='upper right', fontsize=10)

# ── 1b. Healthy vs Diseased pie ───────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
h_imgs = sum(v for k, v in train_counts.items() if 'healthy' in k.lower())
d_imgs = total_train - h_imgs
wedges, texts, autotexts = ax2.pie(
    [h_imgs, d_imgs], labels=['Healthy', 'Diseased'],
    autopct='%1.1f%%', colors=[C['green'], C['red']],
    startangle=140, wedgeprops={'edgecolor':'#0d1117','linewidth':2},
    textprops={'color':'#e6edf3','fontsize':12}
)
for at in autotexts: at.set_color('#0d1117'); at.set_fontweight('bold')
ax2.set_title('Healthy vs Diseased (Train)', color=C['blue'], fontsize=12)

# ── 1c. Top-10 diseased by count ──────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
diseased_cls = [(c, train_counts[c]) for c in CLASS_NAMES if 'healthy' not in c.lower()]
diseased_cls.sort(key=lambda x: x[1], reverse=True)
top10_d = diseased_cls[:10]
names10  = [c[0].replace('___','\n').replace('_',' ')[:28] for c in top10_d]
vals10   = [c[1] for c in top10_d]
gradient = plt.cm.YlOrRd(np.linspace(0.4, 0.9, 10))
ax3.barh(names10[::-1], vals10[::-1], color=gradient, edgecolor='#0d1117')
ax3.set_xlabel('Images')
ax3.set_title('Top-10 Diseased Classes', color=C['blue'], fontsize=12)
ax3.grid(axis='x', alpha=0.3)
for i, v in enumerate(vals10[::-1]):
    ax3.text(v + 20, i, str(v), va='center', fontsize=8, color='#8b949e')

# ── 1d. Train vs Valid split per plant ───────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 2])
plant_train = defaultdict(int)
plant_valid = defaultdict(int)
for c in CLASS_NAMES:
    p = c.split('___')[0]
    plant_train[p] += train_counts[c]
    plant_valid[p] += valid_counts[c]
plant_names = sorted(plant_train.keys())
pt = [plant_train[p] for p in plant_names]
pv = [plant_valid[p] for p in plant_names]
xp = np.arange(len(plant_names))
ax4.bar(xp - 0.2, pt, 0.38, color=C['blue'],   label='Train', alpha=0.85)
ax4.bar(xp + 0.2, pv, 0.38, color=C['purple'], label='Valid', alpha=0.85)
ax4.set_xticks(xp)
ax4.set_xticklabels([p.replace('_',' ') for p in plant_names], rotation=45, ha='right', fontsize=7)
ax4.set_ylabel('Images')
ax4.set_title('Train vs Valid per Plant', color=C['blue'], fontsize=12)
ax4.legend(fontsize=9)
ax4.grid(axis='y', alpha=0.3)

plt.savefig('fig01_eda_dashboard.png', dpi=300, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("✅ Figure 1 saved → fig01_eda_dashboard.png")
max_class = max(train_counts.values())
min_class = min(train_counts.values())

imbalance_ratio = max_class / min_class

print(f"📊 Max Class Images : {max_class}")
print(f"📊 Min Class Images : {min_class}")
print(f"📊 Imbalance Ratio  : {imbalance_ratio:.2f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — Sample Images Gallery (actual leaf photos, 4×6 grid)
# ══════════════════════════════════════════════════════════════════════════════
def load_pil(path, size=224):
    """Load image as RGB PIL, resize."""
    return Image.open(path).convert('RGB').resize((size, size), Image.LANCZOS)

N_ROWS, N_COLS = 4, 6
sample_classes = random.sample(CLASS_NAMES, N_ROWS * N_COLS)

fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(N_COLS * 3.4, N_ROWS * 3.6),
                         facecolor='#0d1117')
fig.suptitle('🌿 Sample Leaf Images — One per Class',
             fontsize=16, color=C['green'], fontweight='bold', y=1.01)

for ax, cls in zip(axes.flat, sample_classes):
    imgs = []
    for ext in ["*.jpg","*.JPG","*.jpeg","*.JPEG","*.png","*.PNG"]:
        imgs.extend(
            list((TRAIN_DIR / cls).glob(ext))
        )

    if len(imgs) == 0:
        print(f"❌ No images found for: {cls}")
        ax.axis('off')
        continue
    img = load_pil(random.choice(imgs))

    ax.imshow(img)
    plant, disease = (cls.split('___') + [''])[:2]
    color = (
        C['green']
        if 'healthy' in disease.lower()
        else C['red']
    )
    ax.set_title(
        f"{plant.replace('_',' ')}\n{disease.replace('_',' ')}",
        fontsize=7.5,color=color,pad=3,fontweight='bold'
    )
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(2.5)
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.savefig('fig02_sample_gallery.png', dpi=300, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f"📊 Classes Displayed : {len(sample_classes)}")
print(f"📷 Images Displayed  : {len(sample_classes)}")
print("✅ Figure 2 saved → fig02_sample_gallery.png")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 3 — RGB Channel Statistics
# ══════════════════════════════════════════════════════════════════════════════
print("Sampling pixels for RGB analysis...")
r_all, g_all, b_all = [], [], []
for cls in random.sample(CLASS_NAMES, min(10, NUM_CLASSES)):
    imgs = list((TRAIN_DIR/cls).glob('*.jpg')) [:8]
    for p in imgs:
        arr = np.array(Image.open(p).convert('RGB').resize((64, 64))) / 255.
        r_all.extend(arr[:,:,0].flatten()); g_all.extend(arr[:,:,1].flatten()); b_all.extend(arr[:,:,2].flatten())

fig, axes = plt.subplots(1, 2, figsize=(16, 5), facecolor='#0d1117')
fig.suptitle('RGB Pixel Intensity Analysis', fontsize=14, color=C['green'], fontweight='bold')

# Histogram
for ch, color, name in zip([r_all, g_all, b_all], [C['red'], C['green'], C['blue']], ['Red','Green','Blue']):
    axes[0].hist(ch, bins=80, alpha=0.6, color=color, label=name, density=True)
axes[0].set_xlabel('Normalised Pixel Intensity'); axes[0].set_ylabel('Density')
axes[0].set_title('Channel Distribution', color=C['blue'])
axes[0].legend(); axes[0].grid(alpha=0.3)

# Box plot
data_bp = [r_all[::50], g_all[::50], b_all[::50]]  # subsample for speed
bp = axes[1].boxplot(data_bp, patch_artist=True, notch=True,
                     medianprops=dict(color='white', linewidth=2))
for patch, color in zip(bp['boxes'], [C['red'], C['green'], C['blue']]):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[1].set_xticklabels(['Red','Green','Blue'])
axes[1].set_ylabel('Pixel Intensity'); axes[1].set_title('Channel Box Plot', color=C['blue'])
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('fig03_rgb_analysis.png', dpi=300, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("✅ Figure 3 saved → fig03_rgb_analysis.png")

---
## 3. 🔄 Data Augmentation Preview <a id='aug'></a>

In [ ]:
# ── Augmentation pipeline (shared) ───────────────────────────────────────────
IMG_SIZE   = 224
MEAN       = [0.485, 0.456, 0.406]   # ImageNet stats
STD        = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomAffine(degrees=20,translate=(0.1, 0.1),scale=(0.9, 1.1),fill=128),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.08),
    transforms.RandomGrayscale(p=0.02),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3,fill=128),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.25,scale=(0.02, 0.15),ratio=(0.3, 3.3)),
])

valid_transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# helper: tensor → displayable numpy
def tensor_to_img(t):
    """Undo normalisation and convert tensor to uint8 numpy."""
    t = t.clone().cpu()
    mean = torch.tensor(MEAN).view(3,1,1)
    std  = torch.tensor(STD ).view(3,1,1)
    t    = t * std + mean                     # de-normalise
    t    = t.clamp(0, 1)
    return (t.permute(1,2,0).numpy() * 255).astype(np.uint8)

print("✅ Transforms defined")
print(f"   Train  : {len(train_transform.transforms)} steps")
print(f"   Valid  : {len(valid_transform.transforms)} steps")
print("\n✅ Augmentations Active")
print("• Horizontal Flip")
print("• Vertical Flip")
print("• Affine Transform")
print("• Color Jitter")
print("• Perspective Distortion")
print("• Random Erasing")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 4 — Augmentation Preview (FIXED — no black images)
# Shows the ORIGINAL leaf + 9 augmented versions side-by-side
# ══════════════════════════════════════════════════════════════════════════════
rand_cls  = random.choice(CLASS_NAMES)
rand_imgs = list((TRAIN_DIR / rand_cls).glob('*.jpg')) + list((TRAIN_DIR / rand_cls).glob('*.JPG'))
orig_path = random.choice(rand_imgs)
orig_pil  = Image.open(orig_path).convert('RGB')          # keep as PIL

N_AUG = 9
fig, axes = plt.subplots(2, 5, figsize=(20, 8), facecolor='#0d1117')
plant_label = rand_cls.replace('___',' — ').replace('_',' ')
fig.suptitle(f'🌿 Augmentation Preview  |  {plant_label}',
             fontsize=14, color=C['green'], fontweight='bold')

# ── Column 0 row 0: original (no normalisation, just resize) ──────────────────
orig_display = orig_pil.resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
axes[0][0].imshow(orig_display)
axes[0][0].set_title('✅ ORIGINAL', color=C['green'], fontsize=11, fontweight='bold')
axes[0][0].set_xticks([]); axes[0][0].set_yticks([])
for sp in axes[0][0].spines.values():
    sp.set_edgecolor(C['green']); sp.set_linewidth(3)

# ── Remaining cells: augmented ────────────────────────────────────────────────
aug_idx = 0
for row in range(2):
    for col in range(5):
        if row == 0 and col == 0:
            continue   # already filled with original
        t   = train_transform(orig_pil)   # returns normalised tensor
        img = tensor_to_img(t)            # de-normalise → numpy uint8
        axes[row][col].imshow(img)
        aug_names = ["Flip","Affine","Color","Perspective","Crop","Mix","Rotate","Shift","Erase"]
        axes[row][col].set_title(aug_names[min(aug_idx, len(aug_names)-1)],color=C['yellow'],fontsize=9)
        axes[row][col].set_xticks([]); axes[row][col].set_yticks([])
        for sp in axes[row][col].spines.values():
            sp.set_edgecolor(C['yellow']); sp.set_linewidth(1.5)
        aug_idx += 1

plt.tight_layout()
plt.savefig('fig04_augmentation.png', dpi=300, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("✅ Figure 4 saved → fig04_augmentation.png")

---
## 4. 📦 DataLoaders <a id='loaders'></a>

In [ ]:
BATCH_SIZE  = 32
NUM_WORKERS = min(8, os.cpu_count())

train_dataset = ImageFolder(TRAIN_DIR, transform=train_transform)
valid_dataset = ImageFolder(VALID_DIR, transform=valid_transform)

# Weighted sampler to handle class imbalance
class_counts = np.array([train_counts[c] for c in train_dataset.classes])
weights_per_class = 1.0 / class_counts
sample_weights = weights_per_class[train_dataset.targets]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=True,persistent_workers=True,prefetch_factor=2, drop_last=True
)
valid_loader = DataLoader(
    valid_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,persistent_workers=True
)

print(f"✅ Train batches : {len(train_loader):,}  ({len(train_dataset):,} images)")
print(f"✅ Valid batches : {len(valid_loader):,}  ({len(valid_dataset):,} images)")
print(f"✅ Classes       : {NUM_CLASSES}")
print(f"✅ Batch size    : {BATCH_SIZE}")
print(f"✅ Workers       : {NUM_WORKERS}")
print("\n📊 Weighted Sampling Enabled")
print(f"Min Class Count : {class_counts.min()}")
print(f"Max Class Count : {class_counts.max()}")
print(f"Imbalance Ratio : {class_counts.max()/class_counts.min():.2f}")

# Verify a batch looks correct
sample_batch, sample_labels = next(iter(train_loader))
print(f"\n🔢 Batch tensor shape : {sample_batch.shape}")
print(f"🔢 Label tensor shape : {sample_labels.shape}")
sample_batch = sample_batch.to(DEVICE,non_blocking=True)
sample_labels = sample_labels.to(DEVICE,non_blocking=True)
print("\n✅ GPU Transfer Successful")

---
## 5. 🧠 Model A — Custom Deep CNN <a id='cnn'></a>

In [ ]:
class ConvBlock(nn.Module):
    """Conv → BN → ReLU → Conv → BN → ReLU → MaxPool → Dropout"""
    def __init__(self, in_ch, out_ch, dropout=0.25, pool=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        ]
        if pool:
            layers += [nn.MaxPool2d(2), nn.Dropout2d(dropout)]
        self.block = nn.Sequential(*layers)

    def forward(self, x): return self.block(x)


class PlantCNN(nn.Module):
    """
    Custom 5-block CNN optimised for leaf disease detection.
    Channels: 3 → 64 → 128 → 256 → 512 → 512 → GAP → FC
    """
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3,   64,  0.20, pool=True),
            ConvBlock(64,  128, 0.25, pool=True),
            ConvBlock(128, 256, 0.25, pool=True),
            ConvBlock(256, 512, 0.30, pool=True),
            ConvBlock(512, 512, 0.00, pool=False),
        )
        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 1024), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(1024, 512), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.head(self.gap(self.features(x)))
    
    def extract_features(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = torch.flatten(x, 1)
        return x

    def get_cam_layer(self):
        """Return last conv layer for Grad-CAM."""
        return self.features[-1].block[3]   # last BN of last ConvBlock


cnn_model = PlantCNN(NUM_CLASSES).to(DEVICE)

total_p     = sum(p.numel() for p in cnn_model.parameters())
trainable_p = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)
print(f"🧠 PlantCNN")
print(f"   Total params     : {total_p:,}")
print(f"   Trainable params : {trainable_p:,}")
print(f"   Device           : {DEVICE}")
print(f"   Classes          : {NUM_CLASSES}")
print("✅ CNN Baseline Ready")

In [ ]:
# ==========================================
# CBAM ATTENTION MODULE
# ==========================================

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)

class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2,1,kernel_size=7,padding=3,bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x,dim=1,keepdim=True)
        max_out, _ = torch.max(x,dim=1,keepdim=True)
        x = torch.cat([avg_out, max_out],dim=1)
        return self.sigmoid(self.conv(x))


class CBAM(nn.Module):

    def __init__(self, channels):
        super().__init__()
        self.ca = ChannelAttention(channels)
        self.sa = SpatialAttention()

    def forward(self, x):
        x = self.ca(x) * x
        x = self.sa(x) * x
        return x

print("="*60)
print("✅ CBAM ATTENTION MODULE LOADED")
print("   Channel Attention Enabled")
print("   Spatial Attention Enabled")
print("="*60)

In [ ]:
dummy = torch.randn(2,128,28,28).to(DEVICE)
cbam_test = CBAM(128).to(DEVICE)
with torch.no_grad():

    output = cbam_test(dummy)

print("Input Shape :", dummy.shape)
print("Output Shape:", output.shape)
print("✅ CBAM Forward Pass Successful")

---
## 6. 🚀 Model B — EfficientNet-B0 (Transfer Learning) <a id='effnet'></a>

In [ ]:
class PlantEfficientNet(nn.Module):
    """
    EfficientNet-B0 backbone (ImageNet pre-trained) with a custom disease-detection head.
    Two-phase training:
      Phase 1 — freeze backbone, train head only.
      Phase 2 — unfreeze last 2 blocks, fine-tune at low LR.
    """
    def __init__(self, num_classes=NUM_CLASSES, pretrained=True):
        super().__init__()
        if HAS_TIMM:
            self.backbone = timm.create_model(
                'efficientnet_b0', pretrained=pretrained, num_classes=0
            )
            feature_dim = self.backbone.num_features
        else:
            weights = tv_models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
            base    = tv_models.efficientnet_b0(weights=weights)
            feature_dim = base.classifier[1].in_features
            base.classifier = nn.Identity()
            self.backbone   = base

        self.head = nn.Sequential(
            nn.Linear(feature_dim, 512), nn.SiLU(), nn.Dropout(0.4),
            nn.Linear(512, 256),         nn.SiLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        self.freeze_backbone()

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad_(False)

    def unfreeze_last_blocks(self, n_blocks=3):
        print(f"🔓 Fine-tuning EfficientNet-B0")
        params = list(self.backbone.parameters())
        n_unfreeze = int(len(params) * 0.25)
        for p in params[-n_unfreeze:]:
            p.requires_grad_(True)
        trainable = sum(
            p.numel()
            for p in self.parameters()
            if p.requires_grad
        )
        print(f"Trainable Parameters: {trainable:,}")
        
    def forward(self, x):
        feats = self.backbone(x)
        if feats.dim() == 4:
            feats = feats.mean([2, 3])  # GAP if backbone still has spatial dims
        return self.head(feats)
    def extract_features(self, x):
        feats = self.backbone(x)
        if feats.dim() == 4:
            feats = feats.mean(
                [2,3]
            )
        return feats


eff_model  = PlantEfficientNet(NUM_CLASSES, pretrained=True).to(DEVICE)
dummy = torch.randn(2,3,IMG_SIZE,IMG_SIZE).to(DEVICE)
with torch.no_grad():
    out = eff_model(dummy)
print(f"Output Shape : {out.shape}")
print("✅ EfficientNet Forward Pass Successful")

total_p    = sum(p.numel() for p in eff_model.parameters())
trainable  = sum(p.numel() for p in eff_model.parameters() if p.requires_grad)
print(f"🚀 PlantEfficientNet-B0")
print(f"   Total params     : {total_p:,}")
print(f"   Trainable params : {trainable:,}  (head only — phase 1)")
print(f"   Device           : {DEVICE}")

In [ ]:
# Model C — EfficientNet-B0 + CBAM

class PlantEfficientNetCBAM(nn.Module):

    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.backbone = timm.create_model("efficientnet_b0",pretrained=True,num_classes=0,global_pool="")
        feature_dim = self.backbone.num_features
        self.cbam = CBAM(feature_dim)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Linear(feature_dim, 512),nn.BatchNorm1d(512),nn.ReLU(),nn.Dropout(0.4),nn.Linear(512, 256),nn.BatchNorm1d(256),nn.ReLU(),nn.Dropout(0.3),nn.Linear(256, num_classes))

        # Freeze backbone initially
        self.freeze_backbone()

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_last_blocks(self, n_blocks=3):

        blocks = self.backbone.blocks
        if hasattr(self.backbone, "blocks"):
            for block in self.backbone.blocks[-n_blocks:]:
                for p in block.parameters():
                    p.requires_grad_(True)

        else:
            params = list(self.backbone.parameters())
            for p in params[-int(len(params)*0.25):]:
                p.requires_grad_(True)

        print(f"🔓 Last {n_blocks} blocks unfrozen")

    def get_cam_layer(self):
        return self.backbone.conv_head

    def forward(self, x):
        x = self.backbone.forward_features(x)
        x = self.cbam(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x
    
cbam_model = PlantEfficientNetCBAM().to(DEVICE)
print("\n" + "="*60)
print("✅ EfficientNet-B0 + CBAM Loaded")
print("="*60)

total_params = sum(
    p.numel()
    for p in cbam_model.parameters()
)
trainable_params = sum(
    p.numel()
    for p in cbam_model.parameters()
    if p.requires_grad
)

print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")

print("="*60)
dummy = torch.randn(2,3,IMG_SIZE,IMG_SIZE).to(DEVICE)
cbam_model.eval()
with torch.no_grad():

    out = cbam_model(dummy)
print(f"Output Shape : {out.shape}")
print("✅ CBAM Forward Pass Successful")

---
## 7. 🏋️ Training Engine <a id='train'></a>

In [ ]:

class LabelSmoothingCrossEntropy(nn.Module):
    """Label smoothing loss for better generalisation."""
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, logits, targets):
        n_cls   = logits.size(-1)
        log_p   = F.log_softmax(logits, dim=-1)
        smooth  = self.smoothing / (n_cls - 1)
        target_p = torch.full_like(log_p, smooth)
        target_p.scatter_(1, targets.unsqueeze(1), 1. - self.smoothing)
        loss = -(target_p * log_p).sum(dim=-1).mean()
        return loss


def train_one_epoch(model, loader, criterion, optimiser, device, scaler=None):
    model.train()
    running_loss, correct, total = 0., 0, 0
    pbar = tqdm(loader, leave=False, desc='  Train')
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimiser.zero_grad()

        if scaler:  # AMP mixed-precision
            with torch.cuda.amp.autocast():
                logits = model(imgs)
                loss   = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimiser)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimiser)
            scaler.update()
        else:
            logits = model(imgs)
            loss   = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimiser.step()

        running_loss += loss.item() * imgs.size(0)
        correct      += (logits.argmax(1) == labels).sum().item()
        total        += imgs.size(0)
        pbar.set_postfix(loss=f'{loss.item():.4f}', acc=f'{correct/total:.4f}')

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0., 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in tqdm(loader, leave=False, desc='  Valid'):
        imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        logits = model(imgs)
        loss   = criterion(logits, labels)
        running_loss += loss.item() * imgs.size(0)
        correct      += (logits.argmax(1) == labels).sum().item()
        total        += imgs.size(0)
        all_preds .extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    precision = precision_score(all_labels,all_preds,average="weighted",zero_division=0)
    recall = recall_score(all_labels,all_preds,average="weighted",zero_division=0)
    f1 = f1_score(all_labels,all_preds,average="weighted",zero_division=0)
    kappa = cohen_kappa_score(all_labels,all_preds)
    mcc = matthews_corrcoef(all_labels,all_preds)
    return (running_loss / total,correct / total,np.array(all_preds),np.array(all_labels),precision,recall,f1,kappa,mcc)


def train_model(model, train_loader, valid_loader, model_name,
                epochs=25, lr=1e-3, save_path=None):
    criterion = LabelSmoothingCrossEntropy(smoothing=0.1)
    optimiser = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                            lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimiser, T_max=epochs, eta_min=lr * 0.01)
    scaler    = torch.cuda.amp.GradScaler() if DEVICE.type == 'cuda' else None

    best_acc = 0.
    best_f1 = 0.
    best_kappa = 0.
    best_mcc = 0.
    best_path    = save_path or f'{model_name.lower().replace(" ","_")}_best.pt'
    patience_cnt = 0
    PATIENCE     = 7

    history = defaultdict(list)

    print(f"\n{'═'*62}")
    print(f"  🏋️  Training {model_name}  |  Epochs={epochs}  LR={lr}")
    print(f"{'═'*62}")

    t_start = time.time()
    for epoch in range(1, epochs + 1):
        t_ep = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimiser, DEVICE, scaler)
        (vl_loss,vl_acc,_,_,precision,recall,f1,kappa,mcc) = evaluate(model,valid_loader,criterion,DEVICE)
        scheduler.step()

        history['tr_loss'].append(tr_loss); history['tr_acc'].append(tr_acc)
        history['vl_loss'].append(vl_loss); history['vl_acc'].append(vl_acc)
        history['precision'].append(precision)
        history['recall'].append(recall)
        history['f1'].append(f1)
        history['kappa'].append(kappa)
        history['mcc'].append(mcc)
        history['lr']     .append(optimiser.param_groups[0]['lr'])

        saved = ''
        if vl_acc > best_acc:
            best_acc     = vl_acc
            best_f1 = f1
            best_kappa = kappa
            best_mcc = mcc
            patience_cnt = 0
            torch.save({'epoch': epoch,'state': model.state_dict(),'best_acc': best_acc,'best_f1': best_f1,
                        'best_kappa': best_kappa,'best_mcc': best_mcc}, best_path)
            saved = '  💾 saved'
        else:
            patience_cnt += 1

        elapsed = time.time() - t_ep
        print(
            f"Ep {epoch:>3}/{epochs} | "
            f"Train={tr_acc*100:6.2f}% | "
            f"Val={vl_acc*100:6.2f}% | "
            f"F1={f1:.4f} | "
            f"Kappa={kappa:.4f} | "
            f"MCC={mcc:.4f} | "
            f"LR={optimiser.param_groups[0]['lr']:.2e}"
            f"{saved}"
        )
        if patience_cnt >= PATIENCE:
            print(f"  ⏹  Early stopping at epoch {epoch}")
            break

    total_time = time.time() - t_start
    print("\n" + "="*70)
    print(f"🏆 Best Accuracy : {best_acc*100:.2f}%")
    print(f"🎯 Best F1 Score : {best_f1:.4f}")
    print(f"📊 Best Kappa    : {best_kappa:.4f}")
    print(f"📈 Best MCC      : {best_mcc:.4f}")
    print(f"⏱ Total Time     : {total_time/60:.1f} min")
    print("="*70) 
    # Restore best weights
    ckpt = torch.load(best_path, map_location=DEVICE)
    model.load_state_dict(ckpt['state'])

    return dict(history), best_acc, total_time

print("✅ Training engine ready")

In [ ]:
# =====================================================
# RESULTS STORAGE
# =====================================================

results = {}

print("✅ Results dictionary created")

---
## 8. 🚂 Train Both Models <a id='run'></a>

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
CNN_EPOCHS      = 30
EFF_EPOCHS_P1   = 12
EFF_EPOCHS_P2   = 15
CBAM_EPOCHS_P1  = 12
CBAM_EPOCHS_P2  = 15
CNN_LR          = 3e-4
EFF_LR_P1       = 1e-3
EFF_LR_P2       = 5e-5
CBAM_LR_P1      = 1e-3
CBAM_LR_P2      = 5e-5

# ─────────────────────────────────────────────────────────────────────────────
# TRAIN MODEL A: Custom CNN
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("🚀 TRAINING MODEL 1 : CUSTOM CNN")
print("="*70)

cnn_history, cnn_best_acc, cnn_time = train_model(
    cnn_model, train_loader, valid_loader,
    model_name='Custom CNN',
    epochs=CNN_EPOCHS, lr=CNN_LR,
    save_path='cnn_best.pt'
)
results["CNN"] = {
    "Accuracy": cnn_best_acc,
    "Training_Time_Min": cnn_time / 60
}
print("✅ CNN Results Saved")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TRAIN MODEL B: EfficientNet-B0 Phase 1 — Head Only
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*70)
print("🚀 TRAINING EFFICIENTNET-B0 : PHASE 1")
print("🔒 Backbone Frozen")
print("="*70)

eff_history1, eff_phase1_acc, eff_phase1_time = train_model(
    eff_model,train_loader,valid_loader,
    model_name='EfficientNet-B0 Phase-1',
    epochs=EFF_EPOCHS_P1,lr=EFF_LR_P1,
    save_path='eff_phase1_best.pt'
)

print("\n" + "="*70)
print("✅ EFFICIENTNET-B0 PHASE 1 COMPLETED")
print(f"🏆 Best Accuracy : {eff_phase1_acc*100:.2f}%")
print(f"⏱ Training Time : {eff_phase1_time/60:.2f} min")
print("="*70)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TRAIN MODEL B: EfficientNet-B0  Phase 2 — fine-tune# 
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("🚀 TRAINING EFFICIENTNET-B0 : PHASE 2")
print("🔓 Fine-Tuning Backbone")
print("="*70)

eff_model.unfreeze_last_blocks(n_blocks=3)
trainable_now = sum(p.numel() for p in eff_model.parameters() if p.requires_grad)
print(f"   Trainable params now: {trainable_now:,}")

eff_history2, eff_best_acc, eff_time = train_model(
    eff_model, train_loader, valid_loader,
    model_name='EfficientNet-B0 Phase-2 (Fine-tune)',
    epochs=EFF_EPOCHS_P2, lr=EFF_LR_P2,
    save_path='eff_best.pt'
)
# Merge phase histories
eff_history = {k: eff_history1[k] + eff_history2[k] for k in eff_history1}

results["EfficientNet-B0"] = {
    "Accuracy": eff_best_acc,
    "Training_Time_Min": eff_time / 60
}
print("✅ EfficientNet-B0 Results Saved")
print("\n" + "="*70)
print("✅ EFFICIENTNET-B0 TRAINING COMPLETED")
print(f"🏆 Best Accuracy : {eff_best_acc*100:.2f}%")
print(f"⏱ Training Time : {eff_time/60:.2f} min")
print("="*70)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TRAIN MODEL C: EfficientNet-B0 + CBAM  Phase 1 — head only
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*70)
print("🚀 STARTING EFFICIENTNET-B0 + CBAM TRAINING")
print("="*70)

cbam_history1, cbam_phase1_acc, cbam_phase1_time = train_model(
    cbam_model,train_loader,valid_loader,
    model_name='EfficientNet-B0 + CBAM Phase-1',
    epochs=CBAM_EPOCHS_P1,lr=CBAM_LR_P1,
    save_path='cbam_phase1_best.pt'
)
print("\n" + "="*70)
print("✅ CBAM PHASE 1 COMPLETED")
print(f"🏆 Best Accuracy : {cbam_phase1_acc*100:.2f}%")
print(f"⏱ Training Time : {cbam_phase1_time/60:.2f} min")
print("="*70)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TRAIN MODEL C: EfficientNet-B0 + CBAM  Phase 2 — fine-tune
# ─────────────────────────────────────────────────────────────────────────────

print("\n🔓 Unfreezing last blocks for CBAM fine-tuning...")

cbam_model.unfreeze_last_blocks(n_blocks=3)

trainable_now = sum(
    p.numel()
    for p in cbam_model.parameters()
    if p.requires_grad
)

print(f"✅ Trainable params now: {trainable_now:,}")

cbam_history2, cbam_best_acc, cbam_time = train_model(
    cbam_model,train_loader,valid_loader,
    model_name='EfficientNet-B0 + CBAM Phase-2 (Fine-tune)',
    epochs=CBAM_EPOCHS_P2,lr=CBAM_LR_P2,
    save_path='cbam_best.pt'
)
print("\n✅ CBAM Fine-Tuning Completed")

cbam_history = {k: cbam_history1[k] + cbam_history2[k] for k in cbam_history1}

results["EfficientNet-B0+CBAM"] = {
    "Accuracy": cbam_best_acc,
    "Training_Time_Min": cbam_time / 60
}
print("✅ CBAM Results Saved")
print("\n" + "="*70)
print("🏆 EFFICIENTNET-B0 + CBAM TRAINING COMPLETED")
print(f"🎯 Best Accuracy : {cbam_best_acc*100:.2f}%")
print(f"⏱ Training Time : {cbam_time/60:.2f} min")
print("="*70)

---
## 9. 📈 Accuracy & Loss Curves <a id='curves'></a>

In [ ]:
def plot_curves(history, title, c_train, c_val, fname, phase2_start=None):
    epochs = list(range(1, len(history['tr_acc']) + 1))
    fig, axes = plt.subplots(1, 4, figsize=(24, 5), facecolor='#0d1117')
    fig.suptitle(title, fontsize=15, color=C['green'], fontweight='bold')

    def add_phase_line(ax):
        if phase2_start:
            ax.axvline(phase2_start+0.5, color=C['orange'], lw=1.5, linestyle=':', alpha=0.8)
            ax.annotate('Fine-tune →', xy=(phase2_start+0.2, ax.get_ylim()[0]*1.01),
                        color=C['orange'], fontsize=8)

    # Accuracy
    axes[0].plot(epochs, [a*100 for a in history['tr_acc']], color=c_train, lw=2.5, label='Train')
    axes[0].plot(epochs, [a*100 for a in history['vl_acc']], color=c_val,   lw=2.5, linestyle='--', label='Validation')
    axes[0].fill_between(epochs,
                         [a*100 for a in history['tr_acc']],
                         [a*100 for a in history['vl_acc']],
                         alpha=0.1, color=c_val)
    best_v = max(history['vl_acc']) * 100
    axes[0].axhline(best_v, color=C['orange'], lw=1, linestyle=':')
    axes[0].annotate(f'Best: {best_v:.2f}%', xy=(len(epochs)*0.55, best_v),
                     color=C['orange'], fontsize=10, va='bottom')
    axes[0].set_title('Accuracy (%)', color=C['blue']); axes[0].set_xlabel('Epoch')
    axes[0].legend(); axes[0].grid(alpha=0.3); add_phase_line(axes[0])

    # Loss
    axes[1].plot(epochs, history['tr_loss'], color=c_train, lw=2.5, label='Train')
    axes[1].plot(epochs, history['vl_loss'], color=c_val,   lw=2.5, linestyle='--', label='Validation')
    axes[1].fill_between(epochs, history['tr_loss'], history['vl_loss'],
                         alpha=0.1, color=C['red'])
    axes[1].set_title('Loss', color=C['blue']); axes[1].set_xlabel('Epoch')
    axes[1].legend(); axes[1].grid(alpha=0.3); add_phase_line(axes[1])

    # F1 Score
    axes[2].plot(epochs,history['f1'],color=C['green'],lw=2.5,label='Validation F1')
    best_f1 = max(history['f1'])
    axes[2].axhline(best_f1,color=C['orange'],linestyle=':',lw=1.5)
    axes[2].annotate(f'Best: {best_f1:.4f}',xy=(len(epochs)*0.55, best_f1),color=C['orange'],fontsize=10)
    axes[2].set_title('F1 Score', color=C['blue'])
    axes[2].set_xlabel('Epoch')
    axes[2].legend()
    axes[2].grid(alpha=0.3)
    add_phase_line(axes[2])

    # Learning rate
    axes[3].plot(epochs, history['lr'], color=C['purple'], lw=2.5)
    axes[3].set_title('Learning Rate', color=C['blue']); axes[3].set_xlabel('Epoch')
    axes[3].set_yscale('log'); axes[3].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(fname, dpi=300, bbox_inches='tight', facecolor='#0d1117')
    plt.show()

    print(
        f"📊 Epochs : {len(epochs)} | "
        f"Best Accuracy : {best_v:.2f}% | "
        f"Best F1 : {best_f1:.4f}")

plot_curves(cnn_history, '📈 Custom CNN — Training Curves',
            C['green'], C['blue'], 'fig05_cnn_curves.png')

plot_curves(eff_history, '📈 EfficientNet-B0 — Training Curves',
            C['purple'], C['orange'], 'fig06_eff_curves.png',
            phase2_start=EFF_EPOCHS_P1)
plot_curves(cbam_history, '📈 EfficientNet-B0 + CBAM — Training Curves',
            C['green'], C['orange'], 'fig07_cbam_curves.png',
            phase2_start=EFF_EPOCHS_P1
)

---
## 10. 📊 Confusion Matrix <a id='cm'></a>

In [ ]:
# ── Run final evaluation ──────────────────────────────────────────────────────
criterion_eval = LabelSmoothingCrossEntropy(smoothing=0.1)

print("Evaluating Custom CNN...")
(_,cnn_val_acc,cnn_preds,cnn_labels,cnn_precision,cnn_recall,cnn_f1,cnn_kappa,cnn_mcc) = evaluate(cnn_model,valid_loader,criterion_eval,DEVICE)

print("Evaluating EfficientNet-B0...")
(_,eff_val_acc,eff_preds,eff_labels,eff_precision,eff_recall,eff_f1,eff_kappa,eff_mcc) = evaluate(eff_model,valid_loader,criterion_eval,DEVICE)

print("Evaluating EfficientNet-B0 + CBAM...")
(_,cbam_val_acc,cbam_preds,cbam_labels,cbam_precision,cbam_recall,cbam_f1,cbam_kappa,cbam_mcc) = evaluate(cbam_model,valid_loader,criterion_eval,DEVICE)

print("✅ CBAM Evaluation Completed")
def plot_confusion_matrix(y_true, y_pred, class_names, title, fname):
    cm      = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    # Custom blue-green cmap for dark theme
    cmap = LinearSegmentedColormap.from_list('plant', ['#0d1117','#1a472a','#3fb950'])

    fig, axes = plt.subplots(1, 2, figsize=(24, 11), facecolor='#0d1117')
    fig.suptitle(f'Confusion Matrix — {title}', fontsize=15, color=C['green'], fontweight='bold')

    for ax, data, subtitle in zip(axes, [cm, cm_norm], ['Raw Counts', 'Normalised (per class)']):
        sns.heatmap(data, ax=ax, cmap=cmap, linewidths=0.2,
                    xticklabels=class_names, yticklabels=class_names,
                    cbar_kws={'shrink': 0.75},
                    annot=(NUM_CLASSES <= 15),
                    fmt=('.0f' if subtitle == 'Raw Counts' else '.2f') if NUM_CLASSES <= 15 else '')
        ax.set_title(subtitle, color=C['blue'], fontsize=12, pad=8)
        ax.set_xlabel('Predicted', color='#8b949e', fontsize=10)
        ax.set_ylabel('Actual',    color='#8b949e', fontsize=10)
        ax.tick_params(axis='x', rotation=90, labelsize=6.5)
        ax.tick_params(axis='y', rotation=0,  labelsize=6.5)

    plt.tight_layout()
    plt.savefig(fname, dpi=300, bbox_inches='tight', facecolor='#0d1117')
    plt.show()
    print(f"✅ Saved → {fname}")

plot_confusion_matrix(cnn_labels, cnn_preds, CLASS_NAMES, 'Custom CNN', 'fig08_cnn_cm.png')
plot_confusion_matrix(eff_labels, eff_preds, CLASS_NAMES, 'EfficientNet-B0', 'fig09_eff_cm.png')
plot_confusion_matrix(cbam_labels,cbam_preds,CLASS_NAMES, 'EfficientNet-B0 + CBAM', 'fig10_cbam_cm.png')

print("\n" + "="*120)
print("📊 OVERALL MODEL PERFORMANCE")
print("="*120)
print(
    f"{'Model':<30}"
    f"{'Accuracy':>12}"
    f"{'Precision':>12}"
    f"{'Recall':>12}"
    f"{'F1':>12}"
    f"{'Kappa':>12}"
    f"{'MCC':>12}"
)
print("-"*120)
print(
    f"{'Custom CNN':<30}"
    f"{cnn_val_acc*100:>12.2f}"
    f"{cnn_precision:>12.4f}"
    f"{cnn_recall:>12.4f}"
    f"{cnn_f1:>12.4f}"
    f"{cnn_kappa:>12.4f}"
    f"{cnn_mcc:>12.4f}"
)
print(
    f"{'EfficientNet-B0':<30}"
    f"{eff_val_acc*100:>12.2f}"
    f"{eff_precision:>12.4f}"
    f"{eff_recall:>12.4f}"
    f"{eff_f1:>12.4f}"
    f"{eff_kappa:>12.4f}"
    f"{eff_mcc:>12.4f}"
)
print(
    f"{'EfficientNet-B0 + CBAM':<30}"
    f"{cbam_val_acc*100:>12.2f}"
    f"{cbam_precision:>12.4f}"
    f"{cbam_recall:>12.4f}"
    f"{cbam_f1:>12.4f}"
    f"{cbam_kappa:>12.4f}"
    f"{cbam_mcc:>12.4f}"
)
print("="*120)

print("\n" + "="*80)
print("📋 CLASSIFICATION REPORT (CBAM MODEL)")
print("="*80)

print(classification_report(cbam_labels,cbam_preds,target_names=CLASS_NAMES,digits=4,zero_division=0))

---
## 11. 📉 Per-Class F1 Analysis <a id='f1'></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 9 — Per-class F1 comparison + precision/recall scatter
# ══════════════════════════════════════════════════════════════════════════════
cnn_rep = classification_report(cnn_labels,cnn_preds,target_names=CLASS_NAMES,output_dict=True)
eff_rep = classification_report(eff_labels,eff_preds,target_names=CLASS_NAMES,output_dict=True)
cbam_rep = classification_report(cbam_labels,cbam_preds,target_names=CLASS_NAMES,output_dict=True)
cnn_f1s = [cnn_rep[c]['f1-score']     for c in CLASS_NAMES]
eff_f1s = [eff_rep[c]['f1-score']     for c in CLASS_NAMES]
cbam_f1s = [cbam_rep[c]['f1-score']   for c in CLASS_NAMES]
cnn_pre = [cnn_rep[c]['precision']    for c in CLASS_NAMES]
eff_pre = [eff_rep[c]['precision']    for c in CLASS_NAMES]
cbam_pre = [cbam_rep[c]['precision']  for c in CLASS_NAMES]
cnn_rec = [cnn_rep[c]['recall']       for c in CLASS_NAMES]
eff_rec = [eff_rep[c]['recall']       for c in CLASS_NAMES]
cbam_rec = [cbam_rep[c]['recall']     for c in CLASS_NAMES]
fig, axes = plt.subplots(1, 2, figsize=(22, 9), facecolor='#0d1117')
fig.suptitle('Per-Class Performance Analysis', fontsize=15, color=C['green'], fontweight='bold')

# Per-class F1 bars
x  = np.arange(NUM_CLASSES)
w  = 0.25
b1 = axes[0].bar(x - w, cnn_f1s, w, color=C['green'],  label='Custom CNN',      alpha=0.85, edgecolor='#0d1117')
b2 = axes[0].bar(x , eff_f1s, w, color=C['purple'], label='EfficientNet-B0', alpha=0.85, edgecolor='#0d1117')
b3 = axes[0].bar(x + w, cbam_f1s, w, color=C['orange'], label='EfficientNet-B0 + CBAM', alpha=0.85, edgecolor='#0d1117')
axes[0].axhline(0.9, color=C['orange'], lw=1, linestyle='--', alpha=0.7, label='90% mark')
axes[0].set_xticks(x)
axes[0].set_xticklabels([c.replace('___','\n').replace('_',' ') for c in CLASS_NAMES],
                         rotation=90, fontsize=6.5, ha='center')
axes[0].set_ylabel('F1 Score'); axes[0].set_ylim(0, 1.08)
axes[0].set_title('F1 Score per Class', color=C['blue'])
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

# Precision vs Recall scatter for EfficientNet
scatter_colors = [C['green'] if 'healthy' in c.lower() else C['red'] for c in CLASS_NAMES]
sc = axes[1].scatter(cbam_rec, cbam_pre, c=[1 if 'healthy' in c.lower() else 0 for c in CLASS_NAMES],
                     cmap='RdYlGn', s=80, alpha=0.85, edgecolors='#0d1117', linewidths=0.5)
axes[1].axhline(0.9, color='#30363d', lw=1, linestyle='--')
axes[1].axvline(0.9, color='#30363d', lw=1, linestyle='--')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_xlim(0, 1.05); axes[1].set_ylim(0, 1.05)
axes[1].set_title('EfficientNet-B0 + CBAM: Precision vs Recall per Class', color=C['blue'])
axes[1].grid(alpha=0.3)
# Annotate outliers (F1 < 0.90)
for i, c in enumerate(CLASS_NAMES):
    if cbam_f1s[i] < 0.90:
        short = c.split('___')[-1].replace('_',' ')[:18]
        axes[1].annotate(short, (cbam_rec[i], cbam_pre[i]),
                         fontsize=6.5, color='#ff7b50',
                         xytext=(5, 5), textcoords='offset points')

plt.tight_layout()
plt.savefig('fig11_f1_analysis.png', dpi=300, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("✅ Figure 11 saved → fig11_f1_analysis.png")

In [ ]:
best_idx = np.argmax(cbam_f1s)
worst_idx = np.argmin(cbam_f1s)
print(
    f"🏆 Best Disease Class : "
    f"{CLASS_NAMES[best_idx]}"
    f" | F1 = {cbam_f1s[best_idx]:.4f}"
)
print(
    f"⚠️ Worst Disease Class : "
    f"{CLASS_NAMES[worst_idx]}"
    f" | F1 = {cbam_f1s[worst_idx]:.4f}"
)

---
## 12. 🖼️ Prediction Gallery <a id='preds'></a>

In [ ]:
def predict_single(model, pil_img, device, return_topk=5):
    """Run inference on a PIL image, return top-k (class, prob) pairs."""
    t    = valid_transform(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(t), dim=1)[0].cpu().numpy()
    top_idx = probs.argsort()[::-1][:return_topk]
    return [(CLASS_NAMES[i], probs[i]) for i in top_idx]


def prediction_gallery(model, model_name, valid_dir, n=16):
    sample_classes = random.sample(CLASS_NAMES, min(n, NUM_CLASSES))
    n_cols = 4; n_rows = (n + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4.2, n_rows * 4.2),
                             facecolor='#0d1117')
    fig.suptitle(f'🔍 Prediction Gallery — {model_name}',
                 fontsize=15, color=C['green'], fontweight='bold', y=1.01)

    model.eval()
    for ax, true_cls in zip(axes.flat, sample_classes):
        imgs = list((valid_dir / true_cls).glob('*.jpg')) + \
               list((valid_dir / true_cls).glob('*.JPG'))
        if not imgs: ax.axis('off'); continue

        pil = Image.open(random.choice(imgs)).convert('RGB')
        preds = predict_single(model, pil, DEVICE)
        pred_cls, conf = preds[0]
        top3_text = "\n".join([
            f"{p[0].split('___')[-1][:12]}:{p[1]*100:.1f}%"
            for p in preds[:3]])
        correct = (pred_cls == true_cls)

        ax.imshow(pil.resize((IMG_SIZE, IMG_SIZE)))
        symbol = '✅' if correct else '❌'
        color  = C['green'] if correct else C['red']

        true_short = true_cls.split('___')[-1].replace('_',' ')
        pred_short = pred_cls.split('___')[-1].replace('_',' ')

        ax.set_title(
            f"{symbol} {pred_short[:20]}\n"
            f"Conf: {conf*100:.1f}%\n"
            f"{top3_text}",
            fontsize=6.5,color=color,pad=4,fontweight='bold')
        for sp in ax.spines.values():
            sp.set_edgecolor(color); sp.set_linewidth(3)
        ax.set_xticks([]); ax.set_yticks([])

    # Hide unused axes
    for ax in axes.flat[n:]:
        ax.axis('off')

    plt.tight_layout()
    fname = f'fig12_{model_name.lower().replace(" ","_")}_gallery.png'
    plt.savefig(fname, dpi=300, bbox_inches='tight', facecolor='#0d1117')
    plt.show()
    print(f"✅ Saved → {fname}")

prediction_gallery(cnn_model, 'Custom CNN',      VALID_DIR, n=16)
prediction_gallery(eff_model, 'EfficientNet-B0', VALID_DIR, n=16)
prediction_gallery(cbam_model,'EfficientNet-B0 + CBAM',VALID_DIR,n=16)

---
## 13. 🔥 Grad-CAM Heatmaps <a id='gradcam'></a>

In [ ]:
class GradCAM:
    """
    Grad-CAM implementation for PyTorch.
    Works with any model by hooking onto a specified target layer.
    """
    def __init__(self, model, target_layer):
        self.model        = model
        self.target_layer = target_layer
        self.gradients    = None
        self.activations  = None
        self._register_hooks()

    def _register_hooks(self):
        def fwd_hook(module, inp, out):
            self.activations = out.detach()
        def bwd_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()
        self.target_layer.register_forward_hook(fwd_hook)
        self.target_layer.register_full_backward_hook(bwd_hook)

    def generate(self, img_tensor, class_idx=None):
        self.model.eval()
        img_tensor = img_tensor.unsqueeze(0).to(DEVICE).requires_grad_(True)
        logits = self.model(img_tensor)
        if class_idx is None:
            class_idx = logits.argmax(dim=1).item()
        self.model.zero_grad()
        logits[0, class_idx].backward()

        # Pool gradients across channels
        grads = self.gradients
        alpha_num = grads.pow(2)
        alpha_denom = (2 * grads.pow(2)+ torch.sum(self.activations * grads.pow(3),dim=(2,3),keepdim=True))
        alpha_denom = torch.where(alpha_denom != 0,alpha_denom,torch.ones_like(alpha_denom))
        alphas = alpha_num / (alpha_denom + 1e-7)
        positive_grads = F.relu(grads)
        weights = torch.sum(alphas * positive_grads,dim=(2,3),keepdim=True)
        cam       = (weights * self.activations).sum(dim=1).squeeze(0)  # [H, W]
        cam = F.relu(cam)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        cam = cam ** 1.3
        return cam.cpu().numpy(), class_idx

def overlay_cam(pil_img, cam, alpha=0.25):

    img_np = np.array(pil_img.resize((IMG_SIZE, IMG_SIZE),Image.LANCZOS))
    cam_up = cv2.resize(cam,(IMG_SIZE, IMG_SIZE),interpolation=cv2.INTER_CUBIC)
    cam_up = cv2.GaussianBlur(cam_up,(5,5),0)
    cam_up = cv2.normalize(cam_up,None,0,1,cv2.NORM_MINMAX)
    cam_up = np.power(cam_up, 1.35)
    hsv = cv2.cvtColor(img_np,cv2.COLOR_RGB2HSV)
    mask = cv2.inRange(hsv,(15,20,20),(120,255,255))
    mask = mask.astype(np.float32)/255.0
    kernel = np.ones((5,5), np.uint8)
    mask = cv2.morphologyEx(mask,cv2.MORPH_CLOSE,kernel)
    cam_up = cam_up * mask
    cam_up = cv2.medianBlur((cam_up * 255).astype(np.uint8),5).astype(np.float32) / 255.0
    heatmap = cv2.applyColorMap(np.uint8(cam_up * 255),cv2.COLORMAP_TURBO)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(img_np,0.80,heatmap,0.20,0)
    return img_np, heatmap, overlay


# ── Hook onto last conv layer ─────────────────────────────────────────────────
# For PlantCNN: last Conv2d inside features
def get_last_conv(model):
    last = None
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            last = m
    return last

cnn_cam = GradCAM(cnn_model,get_last_conv(cnn_model))
eff_cam = GradCAM(eff_model,eff_model.backbone.blocks[-2])
cbam_cam = GradCAM(cbam_model,cbam_model.backbone.blocks[-3])

dummy = torch.randn(1,3,IMG_SIZE,IMG_SIZE).to(DEVICE)
with torch.no_grad():
    out = cbam_model(dummy)
print("✅ CBAM Grad-CAM Hook Verified")
print(f"Output Shape : {out.shape}")

print("="*70)
print("✅ Grad-CAM Ready")
print("="*70)
print("CNN Target Layer    : Last Conv2d")
print("EffNet Target Layer : blocks[-2]")
print("CBAM Target Layer   : blocks[-3]")
print("="*70)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 11 — Grad-CAM Heatmaps (4 samples × 2 models)
# ══════════════════════════════════════════════════════════════════════════════
N_SAMPLES = 4
disease_classes = [c for c in CLASS_NAMES if "healthy" not in c.lower()]
cam_classes = random.sample(disease_classes,N_SAMPLES)
cam_paths   = []
for cls in cam_classes:
    imgs = list((VALID_DIR / cls).glob('*.jpg')) + list((VALID_DIR / cls).glob('*.JPG'))
    if imgs: cam_paths.append((random.choice(imgs), cls))

fig, axes = plt.subplots(len(cam_paths), 7, figsize=(35, len(cam_paths) * 5.5),
                         facecolor='#0d1117')
fig.suptitle('🔥 Grad-CAM Explainability Analysis\n'
    'CNN vs EfficientNet-B0 vs EfficientNet-B0 + CBAM',
             fontsize=15, color=C['green'], fontweight='bold', y=1.01)

col_titles = ['Original', 'CNN Heatmap', 'CNN Overlay', 'Eff Heatmap', 'Eff Overlay','CBAM Heatmap','CBAM Overlay']
for col, ct in enumerate(col_titles):
    axes[0][col].set_title(ct, color=C['blue'], fontsize=10, pad=6, fontweight='bold')

for row, (img_path, cls) in enumerate(cam_paths):
    pil = Image.open(img_path).convert('RGB')
    t   = valid_transform(pil)
    preds = predict_single(cbam_model,pil,DEVICE)
    pred_cls = preds[0][0]
    conf = preds[0][1]
    true_plant = cls.split('___')[0].replace('_', ' ')
    true_disease = cls.split('___')[1].replace('_', ' ')
    pred_plant = pred_cls.split('___')[0].replace('_', ' ')
    pred_disease = pred_cls.split('___')[1].replace('_', ' ')
    if row == 0:
        print("\n")
        print(f"{'Plant':<25} {'Predicted Disease':<35} {'Confidence'}")
        print("="*80)
    print(
        f"{true_plant:<25} "
        f"{pred_disease:<35} "
        f"{conf*100:.2f}%")
    pred_disease = pred_cls.split('___')[-1].replace('_',' ')
    is_correct = (pred_cls == cls)
    border_color = (
        C['green']
        if is_correct
        else C['red'])

    pred_idx = CLASS_NAMES.index(pred_cls)
    cnn_hmap, _ = cnn_cam.generate(t, pred_idx)
    eff_hmap, _ = eff_cam.generate(t, pred_idx)
    cbam_hmap, _ = cbam_cam.generate(t, pred_idx)

    orig, cnn_heat, cnn_over = overlay_cam(pil, cnn_hmap)
    _,    eff_heat, eff_over = overlay_cam(pil, eff_hmap)
    _,    cbam_heat, cbam_over = overlay_cam(pil, cbam_hmap)

    imgs_row = [orig, cnn_heat, cnn_over, eff_heat, eff_over, cbam_heat, cbam_over]
    for col, im in enumerate(imgs_row):
        axes[row][col].imshow(im)
        axes[row][col].set_xticks([])
        axes[row][col].set_yticks([])
        for spine in axes[row][col].spines.values():
            spine.set_edgecolor(border_color)
            spine.set_linewidth(2.5)
    
    # Row label
    disease = cls.split('___')[-1].replace('_', ' ')
    plant   = cls.split('___')[0].replace('_', ' ')
    color   = C['green'] if 'healthy' in disease.lower() else C['red']
    axes[row][0].set_ylabel(
        f"🌿 {plant}\n\n"
        f"✅ True : {disease[:20]}\n"
        f"🤖 Pred : {pred_disease[:20]}\n"
        f"🎯 Conf : {conf*100:.1f}%",
        fontsize=8,color=color,rotation=0,
        labelpad=95,va='center',fontweight='bold')
    
plt.tight_layout()
plt.savefig('fig13_gradcam.png', dpi=300, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("✅ Figure 13 saved → fig13_gradcam.png")

---
## 14. 🏆 Final Summary Dashboard <a id='summary'></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 12 — Summary Dashboard
# ══════════════════════════════════════════════════════════════════════════════
cnn_acc_final = accuracy_score(cnn_labels, cnn_preds)
eff_acc_final = accuracy_score(eff_labels, eff_preds)
cbam_acc_final = accuracy_score(cbam_labels,cbam_preds)
cnn_f1_final  = f1_score(cnn_labels,  cnn_preds,  average='weighted')
eff_f1_final  = f1_score(eff_labels,  eff_preds,  average='weighted')
cbam_f1_final = f1_score(cbam_labels, cbam_preds, average='weighted')
cnn_pre_final = precision_score(cnn_labels, cnn_preds, average='weighted')
eff_pre_final = precision_score(eff_labels, eff_preds, average='weighted')
cbam_pre_final = precision_score(cbam_labels, cbam_preds, average='weighted')
cnn_rec_final = recall_score(cnn_labels, cnn_preds, average='weighted')
eff_rec_final = recall_score(eff_labels, eff_preds, average='weighted')
cbam_rec_final = recall_score(cbam_labels, cbam_preds, average='weighted')


fig = plt.figure(figsize=(20, 12), facecolor='#0d1117')
fig.suptitle('🏆 Final Model Comparison Dashboard',
             fontsize=18, color=C['green'], fontweight='bold', y=0.98)

gs = gridspec.GridSpec(3, 6, figure=fig, hspace=0.6, wspace=0.4)

metrics_pairs = [
    ('Accuracy',  cnn_acc_final*100, eff_acc_final*100, cbam_acc_final*100),
    ('F1 Score',  cnn_f1_final*100, eff_f1_final*100, cbam_f1_final*100),
    ('Precision', cnn_pre_final*100, eff_pre_final*100, cbam_pre_final*100),
    ('Recall',    cnn_rec_final*100, eff_rec_final*100, cbam_rec_final*100),
    ('Kappa',     cnn_kappa*100, eff_kappa*100, cbam_kappa*100),
    ('MCC',       cnn_mcc*100, eff_mcc*100, cbam_mcc*100),
]

for i, (metric, cnn_v, eff_v, cbam_v) in enumerate(metrics_pairs):
    ax = fig.add_subplot(gs[0, i])
    bars = ax.bar(['CNN', 'EfficientNet', 'CBAM'], [cnn_v, eff_v, cbam_v],
                  color=[C['green'], C['purple'], C['orange']], edgecolor='#0d1117',
                  width=0.5, alpha=0.9)
    for bar, v in zip(bars, [cnn_v, eff_v, cbam_v]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
                f'{v:.2f}%', ha='center', va='bottom', fontsize=11,
                color='#e6edf3', fontweight='bold')
    ax.set_ylim(0, 110); ax.set_title(metric, color=C['blue'], fontsize=12)
    ax.grid(axis='y', alpha=0.3); ax.set_ylabel('%')

# Radar chart — relative strengths
ax_radar = fig.add_subplot(gs[1, :3], polar=True)
metrics_labels = ['Accuracy', 'F1 Score', 'Precision', 'Recall', 'Kappa', 'MCC']
cnn_vals_r  = [cnn_acc_final, cnn_f1_final,  cnn_pre_final, cnn_rec_final, cnn_kappa, cnn_mcc]
eff_vals_r  = [eff_acc_final, eff_f1_final,  eff_pre_final, eff_rec_final, eff_kappa, eff_mcc]
cbam_vals_r = [cbam_acc_final, cbam_f1_final, cbam_pre_final, cbam_rec_final, cbam_kappa, cbam_mcc]
N = len(metrics_labels)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]
cnn_vals_r += cnn_vals_r[:1]
eff_vals_r += eff_vals_r[:1]
cbam_vals_r += cbam_vals_r[:1]

ax_radar.plot(angles, cnn_vals_r, color=C['green'],  lw=2.5, label='Custom CNN')
ax_radar.fill(angles, cnn_vals_r, color=C['green'],  alpha=0.2)
ax_radar.plot(angles, eff_vals_r, color=C['purple'], lw=2.5, label='EfficientNet-B0')
ax_radar.fill(angles, eff_vals_r, color=C['purple'], alpha=0.2)
ax_radar.plot(angles, cbam_vals_r, color=C['orange'], lw=2.5, label='EfficientNet-B0 + CBAM')
ax_radar.fill(angles, cbam_vals_r, color=C['orange'], alpha=0.2)
ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(metrics_labels, color='#e6edf3', fontsize=11)
ax_radar.set_ylim(0, 1)
ax_radar.set_facecolor('#161b22')
ax_radar.tick_params(colors='#8b949e')
ax_radar.grid(color='#30363d')
ax_radar.set_title('Performance Radar', color=C['blue'], pad=20, fontsize=12)
ax_radar.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)

# Summary table
ax_tbl = fig.add_subplot(gs[1, 3:])
ax_tbl.axis('off')
table_data = [
    ['Metric',       'Custom CNN',                'EfficientNet-B0',           'EfficientNet-B0 + CBAM'],
    ['Accuracy',     f'{cnn_acc_final*100:.2f}%', f'{eff_acc_final*100:.2f}%', f'{cbam_acc_final*100:.2f}%'],
    ['F1 Score',     f'{cnn_f1_final*100:.2f}%',  f'{eff_f1_final*100:.2f}%',  f'{cbam_f1_final*100:.2f}%'],
    ['Precision',    f'{cnn_pre_final*100:.2f}%', f'{eff_pre_final*100:.2f}%', f'{cbam_pre_final*100:.2f}%'],
    ['Recall',       f'{cnn_rec_final*100:.2f}%', f'{eff_rec_final*100:.2f}%', f'{cbam_rec_final*100:.2f}%'],
    ['Kappa',        f'{cnn_kappa:.4f}',          f'{eff_kappa:.4f}',          f'{cbam_kappa:.4f}'],
    ['MCC',          f'{cnn_mcc:.4f}',            f'{eff_mcc:.4f}',            f'{cbam_mcc:.4f}'],
    ['Parameters',   f'{sum(p.numel() for p in cnn_model.parameters()):,}',
                     f'{sum(p.numel() for p in eff_model.parameters()):,}',
                     f'{sum(p.numel() for p in cbam_model.parameters()):,}'],
    ['Train Time',   f'{cnn_time/60:.1f} min',    f'{eff_time/60:.1f} min',    f'{cbam_time/60:.1f} min'],
]
tbl = ax_tbl.table(cellText=table_data[1:], colLabels=table_data[0],
                   cellLoc='center', loc='center',
                   bbox=[0, 0, 1, 1])
tbl.auto_set_font_size(False); tbl.set_fontsize(10)
for (r, c), cell in tbl.get_celld().items():
    cell.set_facecolor('#161b22' if r > 0 else '#1a472a')
    cell.set_text_props(color='#e6edf3' if r > 0 else C['green'],
                        fontweight='bold' if r == 0 else 'normal')
    cell.set_edgecolor('#30363d')
ax_tbl.set_title('Comparison Table', color=C['blue'], fontsize=12, pad=10)

plt.savefig('fig14_summary_dashboard.png', dpi=300, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("✅ Figure 14 saved → fig14_summary_dashboard.png")

# ── Save models ───────────────────────────────────────────────────────────────
torch.save(cnn_model.state_dict(), 'plant_cnn_final.pt')
torch.save(eff_model.state_dict(), 'plant_efficientnet_final.pt')
torch.save(cbam_model.state_dict(),'plant_cbam_final.pt')

scores = {
    'Custom CNN': cnn_acc_final,
    'EfficientNet-B0': eff_acc_final,
    'EfficientNet-B0 + CBAM': cbam_acc_final
}
winner = max(scores, key=scores.get)
best = scores[winner]

display(HTML(f"""
<div style="background:linear-gradient(135deg,#1a472a,#0d1117);
            padding:24px;border-radius:14px;
            border:2px solid #3fb950;font-family:monospace;margin-top:12px;">
  <div style="color:#3fb950;font-size:22px;font-weight:bold;">🏆 Winner: {winner}</div>
  <div style="color:#e6edf3;font-size:16px;margin-top:10px;">
    Best Validation Accuracy: <b style="color:#58a6ff;">{best*100:.2f}%</b>
  </div>
  <hr style="border-color:#30363d;margin:12px 0;">
  <div style="color:#8b949e;font-size:12px;">
    Saved: plant_cnn_final.pt · plant_efficientnet_final.pt · plant_cbam_final.pt<br>
    Plots: fig01 … fig14
  </div>
</div>
"""))

---
## 15. 🎯 Single Image Inference <a id='inference'></a>

Drop any leaf image path below to run a diagnosis.

In [ ]:
def diagnose_leaf(image_path, model=cbam_model, model_name='EfficientNet-B0 + CBAM', top_k=5):
    """
    Full-featured single-image inference with:
    - Original image display
    - Grad-CAM overlay
    - Top-k confidence bar chart
    """
    pil  = Image.open(image_path).convert('RGB')
    t    = valid_transform(pil)

    # Inference
    model.eval()
    with torch.no_grad():
        logits = model(t.unsqueeze(0).to(DEVICE))
        probs  = torch.softmax(logits, dim=1)[0].cpu().numpy()

    top_idx  = probs.argsort()[::-1][:top_k]
    top_labs = [CLASS_NAMES[i] for i in top_idx]
    top_prob = [probs[i] * 100  for i in top_idx]
    pred_cls = top_labs[0]

    # Grad-CAM
    if model == cnn_model:
        cam_map, _ = cnn_cam.generate(t)
    elif model == eff_model:
        cam_map, _ = eff_cam.generate(t)
    else:
        cam_map, _ = cbam_cam.generate(t)
    _, _, overlay = overlay_cam(pil, cam_map)

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor='#0d1117')
    fig.suptitle(f'🌿 Leaf Diagnosis  |  Model: {model_name}',
                 fontsize=14, color=C['green'], fontweight='bold')

    # Original
    axes[0].imshow(pil.resize((IMG_SIZE, IMG_SIZE)))
    axes[0].text(5,15,f"Prediction: {pred_cls.split('___')[-1].replace('_',' ')}",color='white',fontsize=9,bbox=dict(facecolor='black', alpha=0.7))
    axes[0].set_title('Input Image', color=C['blue'], fontsize=12)
    axes[0].axis('off')

    # Grad-CAM
    axes[1].imshow(overlay)
    axes[1].set_title(f'{model_name} Attention Map', color=C['orange'], fontsize=12)
    axes[1].axis('off')

    # Top-k bar
    short_labs = [f"#{i+1}  " + l.replace('___',' → ').replace('_',' ') for i, l in enumerate(top_labs)]
    bar_colors = [C['green'],C['teal'],C['blue'], C['orange'], C['red']][:top_k]
    axes[2].barh(short_labs[::-1], top_prob[::-1], color=bar_colors[::-1],
                 edgecolor='#0d1117', height=0.6)
    for i, v in enumerate(top_prob[::-1]):
        axes[2].text(v + 0.5, i, f'{v:.1f}%', va='center', color='#e6edf3', fontsize=10)
    axes[2].set_xlim(0, 115); axes[2].set_xlabel('Confidence (%)')
    axes[2].set_title(f'Top-{top_k} Predictions', color=C['blue'], fontsize=12)
    axes[2].grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.savefig('fig15_cbam_inference.png', dpi=300, bbox_inches='tight', facecolor='#0d1117')
    plt.show()

    plant, disease = (pred_cls.split('___') + ['Unknown'])[:2]
    if len(top_labs) > 1:
        second_pred = top_labs[1].replace('___',' → ').replace('_',' ')
        second_conf = top_prob[1]
        prediction_margin = (top_prob[0] -top_prob[1])
    else:
        second_pred = "N/A"
        second_conf = 0
        prediction_margin = 0
    true_class = image_path.split(os.sep)[-2]

    display(HTML(f"""
    <div style="background:linear-gradient(135deg,#161b22,#0d1117);padding:18px;border-radius:12px;
                border:2px solid #3fb950;font-family:monospace;box-shadow:0 0 12px rgba(63,185,80,0.25);">
    <div style="color:#3fb950;font-size:18px;font-weight:bold;margin-bottom:12px;">🌱 Diagnosis Result</div>
    <span style="color:#58a6ff;">Model :</span> <span style="color:#e6edf3;">{model_name}</span><br><br>
    <span style="color:#58a6ff;">Plant :</span> <span style="color:#e6edf3;">{plant.replace('_',' ')}</span><br>
    <span style="color:#58a6ff;">Disease :</span><span style="color:#{'3fb950' if 'healthy' in disease.lower() else 'f85149'};"><b>{disease.replace('_',' ')}</b></span><br>
    <span style="color:#58a6ff;">Confidence :</span><span style="color:#d29922;"> <b>{top_prob[0]:.2f}%</b></span><br>
    <span style="color:#58a6ff;">2nd Prediction :</span><span style="color:#e6edf3;">{second_pred}({second_conf:.2f}%)</span>
    <span style="color:#58a6ff;">Ground Truth :</span><span style="color:#e6edf3;">{true_class.replace('__',' ->').replace('_',' ')}</span><br>
    <span style="color:#58a6ff;">Prediction Margin :</span><span style="color:#e6edf3;">{prediction_margin:.2f}%</span><br>
    <hr style="border-color:#30363d;margin:12px 0;">
    <span style="color:#8b949e;font-size:12px;">  Top-{top_k} predictions generated successfully</span>
    </div>
    """))
    print("\n" + "="*70)
    print("🏆 TOP-5 PREDICTIONS")
    print("="*70)
    for rank, (label, prob) in enumerate(zip(top_labs, top_prob), start=1):
        print(
            f"{rank}. "
            f"{label.replace('___',' → ').replace('_',' ')}"
            f"  |  {prob:.2f}%"
        )
    print("="*70)
    return pred_cls, top_prob[0]
# ── USAGE ────────────────────────────────────────────────────────────────────
# Replace the path below with any leaf image you want to test!
# diagnose_leaf('path/to/your/leaf.jpg')

# Auto-demo with a random validation image
demo_cls  = random.choice(CLASS_NAMES)
demo_imgs = list((VALID_DIR / demo_cls).glob('*.jpg'))
if demo_imgs:
    print(f"🎯 Running demo on: {demo_cls}")
    diagnose_leaf(str(random.choice(demo_imgs)), model=cbam_model, model_name='EfficientNet-B0 + CBAM')
    